# 02 — Exploring one cycle

Before writing a pipeline for 11 cycles, understanding one.

NHANES ships SAS transport (.xpt) files. Each row is one survey participant,
identified by `SEQN`, the ID that links files together. This notebook opens
the 2017-18 demographics and dietary files to locate the variables the project
needs and see how messy they are.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

demo = pd.read_sas(RAW / "DEMO_J.xpt", format="xport")
diet = pd.read_sas(RAW / "DR1TOT_J.xpt", format="xport")

print(f"demo: {demo.shape[0]:,} rows x {demo.shape[1]} columns")
print(f"diet: {diet.shape[0]:,} rows x {diet.shape[1]} columns")

demo: 9,254 rows x 46 columns
diet: 8,704 rows x 168 columns


In [2]:
# 168 columns is too many to eyeball. Filter by name.
caffeine_cols = [c for c in diet.columns if "CAFF" in c.upper()]
print("Caffeine columns:", caffeine_cols)

# Demographics: find the variables for age, gender, income, weights.
for keyword in ["SEQN", "RIDAGEYR", "RIAGENDR", "INDFMPIR", "WTMEC", "WTINT"]:
    matches = [c for c in demo.columns if keyword in c.upper()]
    print(f"{keyword:10} -> {matches}")

Caffeine columns: ['DR1TCAFF']
SEQN       -> ['SEQN']
RIDAGEYR   -> ['RIDAGEYR']
RIAGENDR   -> ['RIAGENDR']
INDFMPIR   -> ['INDFMPIR']
WTMEC      -> ['WTMEC2YR']
WTINT      -> ['WTINT2YR']


In [3]:
# SEQN is the participant ID, the key that links every NHANES file.
df = demo.merge(diet[["SEQN", "DR1TCAFF"]], on="SEQN", how="inner")

# Project scope is US adults.
adults = df[df["RIDAGEYR"] >= 18].copy()

print(f"merged rows:      {len(df):,}")
print(f"adults 18+:       {len(adults):,}")
print(f"missing caffeine: {adults['DR1TCAFF'].isna().sum():,}")
print()
print(adults["DR1TCAFF"].describe())

merged rows:      8,704
adults 18+:       5,533
missing caffeine: 550

count    4.983000e+03
mean     1.409817e+02
std      2.108684e+02
min      5.397605e-79
25%      9.000000e+00
50%      9.400000e+01
75%      1.940000e+02
max      4.320000e+03
Name: DR1TCAFF, dtype: float64


In [4]:
# Dietary analyses use the dietary weight, not the exam weight.
diet_w = diet[["SEQN", "DR1TCAFF", "WTDRD1"]]
dfw = demo[["SEQN", "RIDAGEYR", "RIAGENDR", "INDFMPIR"]].merge(diet_w, on="SEQN", how="inner")

a = dfw[(dfw["RIDAGEYR"] >= 18) & dfw["DR1TCAFF"].notna() & dfw["WTDRD1"].notna()]

unweighted = a["DR1TCAFF"].mean()
weighted = (a["DR1TCAFF"] * a["WTDRD1"]).sum() / a["WTDRD1"].sum()

print(f"unweighted mean: {unweighted:.1f} mg/day")
print(f"weighted mean:   {weighted:.1f} mg/day")
print(f"published 2007-12 benchmark: 169 mg/day")

unweighted mean: 141.0 mg/day
weighted mean:   166.6 mg/day
published 2007-12 benchmark: 169 mg/day


### Why this differs from published estimates

Published NHANES work puts US adult caffeine intake near 169 mg/day
(2007-2012). Four reasons this notebook's figure differs:

1. **Survey weights.** NHANES oversamples certain groups by design, so a raw
   average over-represents them. Published estimates apply weights; the
   unweighted mean above does not.

2. **Which weight.** Dietary analyses require `WTDRD1`, the day-one dietary
   weight, which accounts for non-response to the food interview specifically.
   Using the exam weight `WTMEC2YR` here would be incorrect and is a common
   error in published analyses.

3. **Survey cycle.** Intake has been falling: roughly 175 mg/day in 1999-2000
   down to 142 mg/day by 2011-2012 for the general population. A 2017-18
   figure below the 2007-2012 benchmark is consistent with that trend, not
   evidence of a bug.

4. **Single-day recall.** `DR1TCAFF` is one 24-hour recall, so it captures
   day-to-day variation rather than a person's usual intake. Averaging day 1
   and day 2 (`DR2TCAFF`) reduces that noise.

Population definitions also vary across studies (18+, 19+, or 2+), which
shifts the comparison by a few mg.